# 01 — Data Collection

Fetch article metadata (title, abstract, subjects, Pleiades annotations, etc.)
from the *Archeologia e Calcolatori* (AeC) API and cache it locally as JSONL.

This notebook supports incremental updates: on re-run it only fetches records
newer than the highest `id` already present in the cache.

The cached data feeds notebook `02_ner.ipynb`. The existing `pleiades_uris`
annotations are preserved in the cache but are **not** used to filter or guide
the NER pipeline — they serve only as a gold standard for evaluation in
`06_evaluation.ipynb`.

## CONFIG

Imports and constants: API base URL, the article ID range to fetch, paths for the local cache and output files, and HTTP request settings (User-Agent, timeout).

In [1]:
import json
import logging
from datetime import datetime, timezone
from pathlib import Path

import jsonlines
import pandas as pd
import requests
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("aec_geoparser.data_collection")

API_BASE      = "https://www.archcalc.cnr.it/api"
ID_RANGE_MAX  = 1500          # upper bound; increase for future issues
CACHE_FILE    = Path("../data/cache/articles.jsonl")
META_FILE     = Path("../data/cache/articles.jsonl.meta")
OUT_FILE      = Path("../data/cache/articles_with_abstract.jsonl")
FORCE_REFRESH = False         # set True to re-fetch from scratch

USER_AGENT = "AeC-Geoparser/1.0 (github.com/gmancuso24/AeC_geoparser)"
TIMEOUT_S  = 30


/Users/ijack/Documents/GitHub/AeC_geoparser/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## HTTP session

Helper functions to build a `requests.Session` with a custom User-Agent and to fetch a range of article records from the AeC API.

In [2]:
def make_session() -> requests.Session:
    """Build a requests Session with a descriptive User-Agent header."""
    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT, "Accept": "application/json"})
    return session


def fetch_range(session: requests.Session, first: int, last: int) -> list[dict]:
    """Fetch article records for the inclusive ID range [first, last]."""
    url = f"{API_BASE}/articles/range/{first}/{last}"
    try:
        response = session.get(url, timeout=TIMEOUT_S)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as exc:
        logger.error("Failed to fetch range %s-%s: %s", first, last, exc)
        return []


## Fetch logic

- **First run** (cache absent or `FORCE_REFRESH=True`): fetch the whole `1..ID_RANGE_MAX` range in a single request and write the cache from scratch.
- **Incremental update**: load the existing cache, find the current `max_id`, and fetch only `max_id+1..ID_RANGE_MAX`.

In [3]:
def load_cache(path: Path) -> list[dict]:
    """Load cached article records from a JSONL file."""
    if not path.exists():
        return []
    with jsonlines.open(path) as reader:
        return list(reader)


def write_meta(path: Path, max_id: int, count: int) -> None:
    """Write cache metadata (fetch timestamp, max id, record count) as JSON."""
    meta = {
        "fetched_at": datetime.now(timezone.utc).isoformat(),
        "max_id": max_id,
        "count": count,
    }
    path.write_text(json.dumps(meta, indent=2))


session = make_session()
CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)

cached_records = [] if FORCE_REFRESH else load_cache(CACHE_FILE)

if not cached_records:
    logger.info("Fetching full range 1-%d from %s", ID_RANGE_MAX, API_BASE)
    records = fetch_range(session, 1, ID_RANGE_MAX)
    with jsonlines.open(CACHE_FILE, mode="w") as writer:
        writer.write_all(records)
    max_id = max((r["id"] for r in records), default=0)
    write_meta(META_FILE, max_id=max_id, count=len(records))
    logger.info("Wrote %d records to %s (max_id=%d)", len(records), CACHE_FILE, max_id)
else:
    current_max_id = max(r["id"] for r in cached_records)
    logger.info("Cache present with %d records, max_id=%d", len(cached_records), current_max_id)
    new_records = fetch_range(session, current_max_id + 1, ID_RANGE_MAX)
    if new_records:
        with jsonlines.open(CACHE_FILE, mode="a") as writer:
            writer.write_all(new_records)
    all_records = cached_records + new_records
    new_max_id = max((r["id"] for r in all_records), default=current_max_id)
    write_meta(META_FILE, max_id=new_max_id, count=len(all_records))
    print(f"{len(new_records)} new records added, new max_id={new_max_id}")


2026-06-08 11:44:27,653 [INFO] Cache present with 1440 records, max_id=1500


0 new records added, new max_id=1500


## Dataset summary

Load the cached records into a DataFrame and print summary statistics: total count, how many records have an abstract, how many carry Pleiades URI annotations, language distribution, publication year range, and the average number of Pleiades URIs per annotated article.

In [4]:
records = load_cache(CACHE_FILE)
df = pd.DataFrame(records)

with_abstract = df["abstract"].notna() & (df["abstract"].astype(str).str.strip() != "")
has_pleiades = df["pleiades_uris"].apply(lambda u: isinstance(u, list) and len(u) > 0)

print(f"Total records in cache:           {len(df)}")
print(f"Records with abstract:            {with_abstract.sum()}")
print(f"Records without abstract:         {(~with_abstract).sum()}")
print(f"Records with non-empty pleiades:  {has_pleiades.sum()}")
print(f"Records with empty pleiades:      {(~has_pleiades).sum()}")
print()
print("Language distribution:")
print(df["language"].value_counts(dropna=False).to_string())
print()
print(f"Year range: {df['publication_year'].min()} - {df['publication_year'].max()}")

annotated = df.loc[has_pleiades, "pleiades_uris"]
avg_uris = annotated.apply(len).mean() if len(annotated) else float("nan")
print(f"Average Pleiades URIs per annotated article: {avg_uris:.2f}")


Total records in cache:           1440
Records with abstract:            1358
Records without abstract:         82
Records with non-empty pleiades:  575
Records with empty pleiades:      865

Language distribution:
language
ita    749
eng    557
fre    117
spa     16
ger      1

Year range: 1990 - 2026
Average Pleiades URIs per annotated article: 1.37


## Save filtered set

Keep every record with a non-null, non-empty abstract — regardless of whether `pleiades_uris` is already populated. This is the input for the NER pipeline in `02_ner.ipynb`.

In [5]:
filtered = df.loc[with_abstract].to_dict(orient="records")

with jsonlines.open(OUT_FILE, mode="w") as writer:
    writer.write_all(filtered)

print(f"Saved {len(filtered)} records with abstract to {OUT_FILE}")


Saved 1358 records with abstract to ../data/cache/articles_with_abstract.jsonl
